In [4]:
%%time
import torch
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration

CPU times: user 303 ms, sys: 418 ms, total: 721 ms
Wall time: 2.46 s


In [5]:
%%time
model_name = "BramVanroy/mbart-large-cc25-ft-amr30-en"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

CPU times: user 2.06 s, sys: 336 ms, total: 2.39 s
Wall time: 4.41 s


In [6]:
%%capture
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(253271, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(253271, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True

In [9]:
import re
import string


def linearized_to_penman(linearized):
    # remove AMR start token
    text = linearized.replace("<AMR>", "").strip()

    # map pointers to variables
    pointer_map = {}
    var_names = list(string.ascii_lowercase)
    var_idx = 0

    def replace_pointer(match):
        nonlocal var_idx
        idx = match.group(1)
        if idx not in pointer_map:
            pointer_map[idx] = var_names[var_idx]
            var_idx += 1
        return f"({pointer_map[idx]} / "

    text = re.sub(r"<pointer:(\d+)>", replace_pointer, text)

    # replace relation markers
    text = text.replace("<rel>", " ")
    text = text.replace("</rel>", ")")

    # cleanup spacing
    text = re.sub(r"\s+", " ", text)

    # balance parentheses (simple fix)
    open_par = text.count("(")
    close_par = text.count(")")
    text += ")" * (open_par - close_par)

    return text

In [16]:
%%time
en_sentence = "The boy wants to eat pizza."

tokenizer.src_lang = "en_XX"
inputs = tokenizer(en_sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

generated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["en_XX"], max_length=256
)

print("English AMR:")
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
print(decoded)
penman_amr = linearized_to_penman(decoded)
print(penman_amr.strip())

Both `max_new_tokens` (=512) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English AMR:
<AMR><rel><pointer:0>want-01:ARG0<rel><pointer:1> boy</rel>:ARG1<rel><pointer:2>eat-01:ARG0<pointer:1>:ARG1<rel><pointer:3> pizza</rel></rel></rel>
(a / want-01:ARG0 (b / boy):ARG1 (c / eat-01:ARG0(b / :ARG1 (d / pizza))))
CPU times: user 168 ms, sys: 2.96 ms, total: 171 ms
Wall time: 170 ms


In [17]:
penman_amr.strip()

'(a / want-01:ARG0 (b / boy):ARG1 (c / eat-01:ARG0(b / :ARG1 (d / pizza))))'

In [18]:
%%time
en_sentence = "लड़का पिज़्ज़ा खाना चाहता है।"

tokenizer.src_lang = "hi_IN"
inputs = tokenizer(en_sentence, return_tensors="pt")

inputs = {k: v.to(device) for k, v in inputs.items()}

generated_tokens = model.generate(
    **inputs, forced_bos_token_id=tokenizer.lang_code_to_id["hi_IN"], max_length=256
)

print("Hindi AMR:")
decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
print(decoded)
penman_amr = linearized_to_penman(decoded)
print(penman_amr.strip())

Both `max_new_tokens` (=512) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hindi AMR:
<AMR><rel><pointer:0>want-01:ARG0<rel><pointer:1> woman</rel>:ARG1<rel><pointer:2>eat-01:ARG0<pointer:1>:ARG1<rel><pointer:3> wheat</rel></rel></rel>
(a / want-01:ARG0 (b / woman):ARG1 (c / eat-01:ARG0(b / :ARG1 (d / wheat))))
CPU times: user 172 ms, sys: 4.18 ms, total: 176 ms
Wall time: 176 ms


In [20]:
penman_amr.strip()

'(a / want-01:ARG0 (b / woman):ARG1 (c / eat-01:ARG0(b / :ARG1 (d / wheat))))'